In [3]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np
import pandas as pd

BASE_ROOT = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed")
src_root = BASE_ROOT / "1initNifti"

CT_BACKGROUND = -1024
MR_BACKGROUND = 0
MASK_BACKGROUND = 0

In [4]:
def get_first_nifti(folder: Path):
    if not folder.exists():
        return None
    for pattern in ("*.nii.gz", "*.nii"):
        files = sorted(folder.glob(pattern))
        if files:
            return files[0]
    return None


In [5]:
def collect_metadata(src_root: Path) -> pd.DataFrame:
    rows = []

    for case_dir in sorted(src_root.iterdir()):
        if not case_dir.is_dir():
            continue

        case_id = case_dir.name

        ct_in = get_first_nifti(case_dir / "CT_reg")
        mr_in = get_first_nifti(case_dir / "MR")
        mask_in = get_first_nifti(case_dir / "masks")

        def process_one(modality: str, in_path: Path):
            img = sitk.ReadImage(str(in_path))
            arr = sitk.GetArrayFromImage(img)  # (z, y, x)
            z, y, x = arr.shape

            sx, sy, sz = img.GetSpacing()  # (sx, sy, sz) in physical coords

            rows.append({
                "case_id": case_id,
                "modality": modality,
                "filename": in_path.name,
                "z": z,
                "y": y,
                "x": x,
                "spacing_x": sx,
                "spacing_y": sy,
                "spacing_z": sz,
            })

        if ct_in is not None:
            process_one("CT", ct_in)
        if mr_in is not None:
            process_one("MR", mr_in)
        if mask_in is not None:
            process_one("MASK", mask_in)

    return pd.DataFrame(rows)


In [4]:
df = collect_metadata(src_root)
df.head()


,case_id,modality,filename,z,y,x,spacing_x,spacing_y,spacing_z
0,AB_1ABC100,CT,AB_1ABC100_CT_reg.nii.gz,90,434,441,1.0,1.0,3.0
1,AB_1ABC100,MR,AB_1ABC100_MR.nii.gz,90,434,441,1.0,1.0,3.0
2,AB_1ABC100,MASK,AB_1ABC100_mask.nii.gz,90,434,441,1.0,1.0,3.0
3,AB_1ABC116,CT,AB_1ABC116_CT_reg.nii.gz,49,444,443,1.0,1.0,3.0
4,AB_1ABC116,MR,AB_1ABC116_MR.nii.gz,49,444,443,1.0,1.0,3.0


In [1]:
df.to_csv(BASE_ROOT / "dataset_shape_spacing_stats.csv", index=False)

NameError: name 'df' is not defined

In [11]:
df = pd.read_csv(BASE_ROOT / "dataset_shape_spacing_stats.csv")
df.head()

df.sample(20)

,case_id,modality,filename,z,y,x,spacing_x,spacing_y,spacing_z
434,brain_1BB189,MASK,brain_1BB189_mask.nii.gz,198,230,218,1.0,1.0,1.0
83,TH_1THA060,MASK,TH_1THA060_mask.nii.gz,101,398,551,1.0,1.0,3.0
845,pelvis_1PA118,MASK,pelvis_1PA118_mask.nii.gz,146,271,466,1.0,1.0,2.5
277,brain_1BB007,MR,brain_1BB007_MR.nii.gz,200,236,211,1.0,1.0,1.0
918,pelvis_1PA159,CT,pelvis_1PA159_CT_reg.nii.gz,118,307,461,1.0,1.0,2.5
368,brain_1BB083,MASK,brain_1BB083_mask.nii.gz,191,235,210,1.0,1.0,1.0
1094,pelvis_1PC058,MASK,pelvis_1PC058_mask.nii.gz,92,319,437,1.0,1.0,2.5
896,pelvis_1PA148,MASK,pelvis_1PA148_mask.nii.gz,136,284,480,1.0,1.0,2.5
440,brain_1BB200,MASK,brain_1BB200_mask.nii.gz,192,239,220,1.0,1.0,1.0
754,pelvis_1PA076,MR,pelvis_1PA076_MR.nii.gz,124,353,479,1.0,1.0,2.5


In [27]:
df["body_part"] = df["case_id"].str.split("_").str[0]

df.groupby("body_part")[["y", "x"]].describe()

y                                                              \
           count        mean        std    min     25%    50%     75%    max   
body_part                                                                      
AB          21.0  389.428571  38.806664  339.0  350.00  393.0  434.00  444.0   
HN          39.0  270.615385  22.707077  241.0  252.00  265.0  295.00  306.0   
TH          24.0  401.375000   5.207123  390.0  400.25  402.5  404.50  407.0   
brain      540.0  251.233333  14.992429  224.0  239.00  248.0  264.25  284.0   
pelvis     540.0  299.661111  30.240072  248.0  278.75  294.0  318.00  410.0   

               x                                                             
           count        mean        std    min     25%    50%    75%    max  
body_part                                                                    
AB          21.0  462.142857  41.502151  426.0  436.00  441.0  514.0  536.0  
HN          39.0  266.846154  21.592565  242.0  250.00  260.0  295.0  303.0  
TH          24.0  548.375000   4.301794  544.0  544.75  547.0  551.5  556.0  
brain      540.0  219.644444  16.738285  173.0  208.75  219.0  228.0  280.0  
pelvis     540.0  461.072222  45.581222  390.0  429.00  448.5  482.0  586.0

In [28]:
df.groupby("body_part")[["spacing_z", "spacing_y", "spacing_x"]].describe()

spacing_z                                    spacing_y       ...  \
              count mean  std  min  25%  50%  75%  max     count mean  ...   
body_part                                                              ...   
AB             21.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      21.0  1.0  ...   
HN             39.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      39.0  1.0  ...   
TH             24.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      24.0  1.0  ...   
brain         540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0     540.0  1.0  ...   
pelvis        540.0  2.5  0.0  2.5  2.5  2.5  2.5  2.5     540.0  1.0  ...   

                    spacing_x                                     
           75%  max     count mean  std  min  25%  50%  75%  max  
body_part                                                         
AB         1.0  1.0      21.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
HN         1.0  1.0      39.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
TH         1.0  1.0      24.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
brain      1.0  1.0     540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
pelvis     1.0  1.0     540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  

[5 rows x 24 columns]

In [26]:
def extract_hosp_char(case_id: str, body_part: str) -> str:
    s = case_id.upper()
    
    # mapping from body_part to the prefix we search in the case_id
    prefix_map = {
        "pelvis": "P",
        "brain": "B",
        "TH": "TH",
        "HN": "HN",
        "AB": "AB",
    }

    prefix = prefix_map.get(body_part, body_part)  # fallback: use body_part itself
    idx = s.rfind(prefix)

    pos = idx + len(prefix)
    if pos < len(s):
        hospital = s[pos]
        if body_part in ("pelvis", "brain"):
            return "2023" + hospital
        else:
            return "2025" + hospital
    else:
        return ""

df["hosp_char"] = df.apply(
    lambda row: extract_hosp_char(row["case_id"], row["body_part"]),
    axis=1,
)




In [25]:
df.groupby(["hosp_char", "body_part"])["spacing_x"].describe()

count  mean  std  min  25%  50%  75%  max
hosp_char body_part                                           
20230     brain      126.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20231     brain       48.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20232     brain        6.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023A     brain      180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis     360.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023C     brain      180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis     180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025A     TH          24.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025C     AB          21.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          HN          39.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0